In [ ]:
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("projetoanalisecinema").getOrCreate()

In [ ]:
# Leitura CSV Filme B

df_salas = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/Raking de estado por sala - Página1.csv",header=True,inferSchema=True)

df_publico = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/Ranking de estados por público - Filme B - Página1.csv",header=True,inferSchema=True)

In [ ]:
# Conferir CSV

df_salas.show(5, truncate=False)
df_salas.printSchema()

df_publico.show(5, truncate=False)
df_publico.printSchema()

In [ ]:
# Conferir os prints

print(df_publico.columns)
print(df_salas.columns)

In [ ]:
# Padronização de colunas

df_salas = df_salas \
    .withColumnRenamed("SALAS 3D", "SALAS_3D") \
    .withColumnRenamed("SALAS 2D", "SALAS_2D") \
    .withColumnRenamed("TOTAL DE SALAS", "TOTAL_DE_SALAS")

In [ ]:
# Conferir a padronização das colunas

print(df_salas.columns)

In [ ]:
# Limpeza de números

from pyspark.sql.functions import regexp_replace, col

df_publico = df_publico \
    .withColumn("PUBLICO", regexp_replace(col("PUBLICO"), r"\.", "")) \
    .withColumn("PUBLICO", col("PUBLICO").cast("long")) \
    .withColumn("RENDA", regexp_replace(col("RENDA"), r"\.", "")) \
    .withColumn("RENDA", regexp_replace(col("RENDA"), ",", ".")) \
    .withColumn("RENDA", col("RENDA").cast("double")) \
    .withColumn("PMI", regexp_replace(col("PMI"), ",", ".")) \
    .withColumn("PMI", col("PMI").cast("double"))

df_salas = df_salas \
    .withColumn("SALAS_2D", regexp_replace(col("SALAS_2D"), ",", ".")) \
    .withColumn("SALAS_2D", col("SALAS_2D").cast("double"))

In [ ]:
# Conferir a limpeza

df_publico.printSchema()
df_salas.printSchema()

df_publico.show(5)
df_salas.show(5)

In [ ]:
# Salvar os dataframes tratados em formato parquet

df_salas.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_salas_tratado")

df_publico.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_publico_tratado")

In [ ]:
# Leitura CSV ANCINE

df_bilheteria_dezembro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaDezembro2025.csv",header=True,inferSchema=True)

df_bilheteria_janeiro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaJaneiro2026.csv",header=True,inferSchema=True)

df_bilheteria_fevereiro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaFevereiro2026.csv",header=True,inferSchema=True)

In [ ]:
# Conferir CSV

df_bilheteria_dezembro.show(5, truncate=False)
df_bilheteria_dezembro.printSchema()

df_bilheteria_janeiro.show(5, truncate=False)
df_bilheteria_janeiro.printSchema()

df_bilheteria_fevereiro.show(5, truncate=False)
df_bilheteria_fevereiro.printSchema()

In [ ]:
# Conversão de data string para tipo date - Dezembro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_dezembro = df_bilheteria_dezembro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Conversão de data string para tipo date - Janeiro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_janeiro = df_bilheteria_janeiro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Conversão de data string para tipo date - Fevereiro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_fevereiro = df_bilheteria_fevereiro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Verificação da conversão das datas

df_bilheteria_dezembro.show(5, truncate=False)
df_bilheteria_dezembro.printSchema()

df_bilheteria_janeiro.show(5, truncate=False)
df_bilheteria_janeiro.printSchema()

df_bilheteria_fevereiro.show(5, truncate=False)
df_bilheteria_fevereiro.printSchema()

In [ ]:
# Juntar os CSVS em um único DataFrame

df_ancine = df_bilheteria_dezembro.unionByName(df_bilheteria_janeiro).unionByName(df_bilheteria_fevereiro)

In [ ]:
# Verificar junção

df_ancine.count()

In [ ]:
# Verificar colunas

df_ancine.printSchema()
df_ancine.show(5, truncate=False)
df_ancine.count()

In [ ]:
# Verificar valores nulos no DF Ancine

from pyspark.sql.functions import col, sum

df_ancine.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_ancine.columns
]).show()

In [ ]:
# Verificar as linhas nulas antes de removê-las

df_ancine.filter(
    col("TITULO_BRASIL").isNull() & col("TITULO_ORIGINAL").isNull()
).show(5, truncate=False)

In [ ]:
# Criar coluna Título Principal

from pyspark.sql.functions import col, coalesce

df_ancine = df_ancine.withColumn(
    "TITULO_PRINCIPAL",
    coalesce(col("TITULO_BRASIL"), col("TITULO_ORIGINAL"))
)

In [ ]:
# Remover linhas vazias

df_ancine = df_ancine.dropna(subset=[
    "TITULO_PRINCIPAL",
    "DATA_EXIBICAO",
    "PUBLICO"
])

In [ ]:
# Deixar apenas Título Principal

df_ancine = df_ancine.drop("TITULO_BRASIL", "TITULO_ORIGINAL")

In [ ]:
# Verificar esses ajustes

df_ancine.printSchema()
df_ancine.show(5, truncate=False)
df_ancine.count()

In [ ]:
# Salvar dataframe final Ancine

df_ancine.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/dados_tratados/ancine_tratado")

In [ ]:
# Análise público por UF

df_publico_uf = df_ancine.groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_TOTAL") \
    .orderBy("PUBLICO_TOTAL", ascending=False)

df_publico_uf.show()

In [ ]:
# Análise salas por UF

df_salas_uf = df_ancine.select("UF", "REGISTRO_SALA") \
    .distinct() \
    .groupBy("UF") \
    .count() \
    .withColumnRenamed("count", "QTD_SALAS") \
    .orderBy("QTD_SALAS", ascending=False)

df_salas_uf.show()

In [ ]:
# Juntar público por UF e salas por UF

df_analise_uf = df_publico_uf.join(df_salas_uf, on="UF")

In [ ]:
# Verificar junção público por UF e salas por UF

df_analise_uf.printSchema()
df_analise_uf.show()
df_analise_uf.count()

In [ ]:
# Conferir valores nulos no df_analise_uf

from pyspark.sql.functions import sum, col

df_analise_uf.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_analise_uf.columns
]).show()

In [ ]:
# Remover linhas 0

from pyspark.sql.functions import col

df_analise_uf = df_analise_uf.filter(col("UF") != "0")

In [ ]:
# Conferir remoção

df_analise_uf.select("UF").distinct().show()

In [ ]:
# Criar coluna público por sala

df_analise_uf = df_analise_uf.withColumn(
    "PUBLICO_POR_SALA",
    col("PUBLICO_TOTAL") / col("QTD_SALAS")
)

In [ ]:
# Ordenar a coluna público por sala

df_analise_uf.orderBy("PUBLICO_POR_SALA", ascending=False).show()

In [ ]:
# Conferir as colunas do df_analise_uf

df_analise_uf.columns

In [ ]:
# Evolução do público por mês

df_publico_mes = df_ancine.groupBy("MES") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_TOTAL") \
    .orderBy("MES")

df_publico_mes.show()

In [ ]:
# Público por UF em fevereiro

from pyspark.sql.functions import col

df_fev_uf = df_ancine.filter(col("MES") == 2) \
    .groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_FEVEREIRO") \
    .orderBy("PUBLICO_FEVEREIRO", ascending=False)

df_fev_uf.show()

In [ ]:
# Semana do Cinema (R$10)

df_semana_cinema = df_ancine.filter(
    (col("DATA_EXIBICAO") >= "2026-02-08") &
    (col("DATA_EXIBICAO") <= "2026-02-14")
)

In [ ]:
# Público na semana so Cinema (R$10)

df_semana_cinema.groupBy("UF") \
    .sum("PUBLICO") \
    .orderBy("sum(PUBLICO)", ascending=False) \
    .show()

In [ ]:
# Participação público da semana do cinema por UF

from pyspark.sql.functions import col, sum, round

# total da semana
total_semana = df_semana_cinema.agg(
    sum("PUBLICO").alias("TOTAL")
).collect()[0]["TOTAL"]

# participação
df_participacao = df_semana_cinema.groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_SEMANA") \
    .withColumn(
        "PARTICIPACAO_%",
        round((col("PUBLICO_SEMANA") / total_semana) * 100, 2)
    ) \
    .orderBy("PARTICIPACAO_%", ascending=False)

df_participacao.show()

In [ ]:
# Uma linha por municipio no df_salas

from pyspark.sql.functions import sum

df_salas_agg = df_salas.groupBy("UF", "MUNICIPIO") \
    .agg(
        sum("SALAS_2D").alias("SALAS_2D"),
        sum("SALAS_3D").alias("SALAS_3D")
    )

In [ ]:
# Uma linha por municipio o df_publico

df_publico_agg = df_publico.groupBy("UF", "MUNICIPIO") \
    .agg(
        sum("RENDA").alias("RENDA"),
        sum("PMI").alias("PMI")
    )

In [ ]:
# Join df_salas e df_publico

df_filmeb = df_publico_agg.join(
    df_salas_agg,
    on=["UF", "MUNICIPIO"],
    how="left"
)

In [ ]:
# Verificar join df_salas e df_publico

df_filmeb.show()
df_filmeb.count()

In [ ]:
# Criar variável total de salas

from pyspark.sql.functions import col

df_filmeb = df_filmeb.withColumn(
    "TOTAL_SALAS",
    col("SALAS_2D") + col("SALAS_3D")
)

In [ ]:
# Verificar PMI e RENDA

df_filmeb.select("PMI", "RENDA").show()

In [ ]:
# Agrupar PMI por faixa de preço

from pyspark.sql.functions import when

df_filmeb = df_filmeb.withColumn(
    "FAIXA_PRECO",
    when(col("PMI") < 10, "Até 10")
    .when((col("PMI") >= 10) & (col("PMI") < 20), "10 a 20")
    .when((col("PMI") >= 20) & (col("PMI") < 30), "20 a 30")
    .otherwise("30+")
)

In [ ]:
# Analisar renda por faixa de preço

df_filmeb.groupBy("FAIXA_PRECO") \
    .sum("RENDA") \
    .orderBy("FAIXA_PRECO") \
    .show()

In [ ]:
# Fazer renda média faixa de preço

from pyspark.sql.functions import avg

df_filmeb.groupBy("FAIXA_PRECO") \
    .agg(avg("RENDA").alias("RENDA_MEDIA")) \
    .orderBy("FAIXA_PRECO") \
    .show()

In [ ]:
# Correlação PMI e Renda   - # perto de 0 -> fraca ; # perto de 1 -> alta (relação positiva)

df_filmeb.stat.corr("PMI", "RENDA")

In [ ]:
# Renda por quantidade de salas

df_filmeb.stat.corr("TOTAL_SALAS", "RENDA")

In [ ]:
# Sala vs público

from pyspark.sql.functions import sum

# Público por UF
df_publico_uf = df_ancine.groupBy("UF") \
    .agg(sum("PUBLICO").alias("PUBLICO_TOTAL"))

# Salas por UF
df_salas_uf = df_ancine.select("UF", "REGISTRO_SALA") \
    .distinct() \
    .groupBy("UF") \
    .count() \
    .withColumnRenamed("count", "QTD_SALAS")

# Join
df_corr_publico_uf = df_publico_uf.join(df_salas_uf, "UF")

# Correlação
df_corr_publico_uf.stat.corr("QTD_SALAS", "PUBLICO_TOTAL")

In [ ]:
# Correlação PMI vs total de salas

df_filmeb.stat.corr("PMI", "TOTAL_SALAS")

In [ ]:
# Semana do cinema vs estrutura

df_semana = df_participacao.join(
    df_analise_uf.select("UF", "QTD_SALAS"),
    on="UF"
)

df_semana.stat.corr("QTD_SALAS", "PARTICIPACAO_%")

In [ ]:
# Converter parquet Ancine para CSV

df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/dados_tratados/ancine_tratado")

df.coalesce(1).write.mode("overwrite").csv("/content/dados_csv", header=True)

In [ ]:
# Converter parquet Film B pra CSV

df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_salas_tratado")
df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_publico_tratado")

df_filmeb.coalesce(1).write.mode("overwrite").csv("/content/filmeb_csv", header=True)


In [ ]:
spark.stop()